In [86]:
# For group members
# Run this cell once. 
# If your env already has these packages, it may say they are already installed.
# otherwise it will install 
!pip install torch torchvision torchaudio
!pip install kagglehub

from pathlib import Path
import os
import random
import json
import shutil
import kagglehub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

#print("Torch version:", torch.__version__)
SEED = 67
#just ensuring our machine is using the CPU instead of GPU
#just tells torth where to do all our math
device = torch.device("cpu")
#double checking
print("Using device:", device)

Using device: cpu


In [87]:
KAGGLE_DATASET = "ahemateja19bec1025/traffic-sign-dataset-classification"

kaggle_input_path = Path("/kaggle/input/traffic-sign-dataset-classification")

if kaggle_input_path.exists():
    dataset_root = kaggle_input_path
    print("Using Kaggle input dataset:", dataset_root)
else:
    dataset_root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    print("Downloaded dataset to:", dataset_root)

Downloaded dataset to: /Users/thomas/.cache/kagglehub/datasets/ahemateja19bec1025/traffic-sign-dataset-classification/versions/2


In [88]:
labels_df = pd.read_csv("labels.csv")
labels_df.head()

,ClassId,Name
0,0,Speed limit (5km/h)
1,1,Speed limit (15km/h)
2,2,Speed limit (30km/h)
3,3,Speed limit (40km/h)
4,4,Speed limit (50km/h)


In [89]:
#Used Chat GPT to find a good way to group the signs instead of using 58 classes
#we use 4 and dropped all the classes named "unknown"
group_map = {
    # 0: Speed limit
    0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 18: 0, 19: 0,

    # 1: Prohibited / No signs
    8: 1, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1, 14: 1, 15: 1,
    16: 1, 17: 1, 54: 1, 55: 1,

    # 2: Mandatory / Direction
    20: 2, 21: 2, 22: 2, 23: 2, 24: 2, 25: 2, 26: 2, 27: 2,
    29: 2, 31: 2, 43: 2, 44: 2, 53: 2,

    # 3: Warning / Hazard / Road condition
    28: 3, 30: 3, 32: 3, 33: 3, 34: 3, 35: 3, 36: 3, 37: 3,
    38: 3, 39: 3, 46: 3, 47: 3, 48: 3, 50: 3, 51: 3,
}

unknown_ids = [40, 41, 42, 45, 49, 52, 56, 57]

class_names = {
    0: "Speed limit",
    1: "Prohibited / No sign",
    2: "Mandatory / Direction",
    3: "Warning / Hazard",
}

# Drop unknown rows from the label table and add grouped labels.
labels_clean = labels_df[~labels_df["ClassId"].isin(unknown_ids)].copy()
labels_clean["GroupedClassId"] = labels_clean["ClassId"].map(group_map)
labels_clean["GroupedName"] = labels_clean["GroupedClassId"].map(class_names)

print("Original number of label rows:", len(labels_df))
print("Number after dropping unknown labels:", len(labels_clean))
print("Missing grouped labels:", labels_clean["GroupedClassId"].isna().sum())

labels_clean.head(15)

Original number of label rows: 58
Number after dropping unknown labels: 50
Missing grouped labels: 0


,ClassId,Name,GroupedClassId,GroupedName
0,0,Speed limit (5km/h),0,Speed limit
1,1,Speed limit (15km/h),0,Speed limit
2,2,Speed limit (30km/h),0,Speed limit
3,3,Speed limit (40km/h),0,Speed limit
4,4,Speed limit (50km/h),0,Speed limit
5,5,Speed limit (60km/h),0,Speed limit
6,6,Speed limit (70km/h),0,Speed limit
7,7,speed limit (80km/h),0,Speed limit
8,8,Dont Go straight or left,1,Prohibited / No sign
9,9,Dont Go straight or Right,1,Prohibited / No sign


In [90]:
# Show how many original sign labels are in each new group.
labels_clean.groupby(["GroupedClassId", "GroupedName"])["ClassId"].count().rename("Number of original labels")

GroupedClassId  GroupedName          
0               Speed limit              10
1               Prohibited / No sign     12
2               Mandatory / Direction    13
3               Warning / Hazard         15
Name: Number of original labels, dtype: int64

In [91]:
def find_class_folder_root(root: Path) -> Path:
    # Find the folder whose immediate subfolders are numeric class labels.
    preferred = [
        root / "traffic_Data" / "DATA",
        root / "traffic_Data",
        root / "DATA",
        root,
    ]
    
    for p in preferred:
        if p.exists() and p.is_dir():
            numeric_children = [child for child in p.iterdir() if child.is_dir() and child.name.isdigit()]
            if len(numeric_children) >= 5:
                return p
    
    candidates = []
    for p in root.rglob("*"):
        if p.is_dir():
            numeric_children = [child for child in p.iterdir() if child.is_dir() and child.name.isdigit()]
            if len(numeric_children) >= 5:
                candidates.append((len(numeric_children), p))
    
    if not candidates:
        raise FileNotFoundError("Could not find a folder with numeric class subfolders.")
    
    candidates.sort(reverse=True)
    return candidates[0][1]

DATA_DIR = find_class_folder_root(dataset_root)
print("Using image class folder:", DATA_DIR)
print("Example class folders:", sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir() and p.name.isdigit()])[:10])

Using image class folder: /Users/thomas/.cache/kagglehub/datasets/ahemateja19bec1025/traffic-sign-dataset-classification/versions/2/traffic_Data/DATA
Example class folders: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17']


In [92]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
label_lookup = dict(zip(labels_df["ClassId"], labels_df["Name"]))

rows = []
for class_folder in sorted([p for p in DATA_DIR.iterdir() if p.is_dir() and p.name.isdigit()], key=lambda p: int(p.name)):
    original_class_id = int(class_folder.name)
    
    # This line drops all unknown classes and any class not included in group_map.
    if original_class_id not in group_map:
        continue
    
    grouped_class_id = group_map[original_class_id]
    for img_path in class_folder.rglob("*"):
        if img_path.suffix.lower() in image_extensions:
            rows.append({
                "image_path": str(img_path),
                "ClassId": original_class_id,
                "OriginalName": label_lookup.get(original_class_id, "Unknown label name"),
                "GroupedClassId": grouped_class_id,
                "GroupedName": class_names[grouped_class_id],
            })

image_df = pd.DataFrame(rows)

if image_df.empty:
    raise ValueError("No images were found. Check DATA_DIR and the dataset folder structure.")

print("Total usable images after dropping unknown classes:", len(image_df))
print("Grouped class counts:")
print(image_df["GroupedName"].value_counts())

image_df.head()

Total usable images after dropping unknown classes: 3870
Grouped class counts:
GroupedName
Prohibited / No sign     1258
Warning / Hazard         1062
Speed limit              1032
Mandatory / Direction     518
Name: count, dtype: int64


,image_path,ClassId,OriginalName,GroupedClassId,GroupedName
0,/Users/thomas/.cache/kagglehub/datasets/ahemat...,0,Speed limit (5km/h),0,Speed limit
1,/Users/thomas/.cache/kagglehub/datasets/ahemat...,0,Speed limit (5km/h),0,Speed limit
2,/Users/thomas/.cache/kagglehub/datasets/ahemat...,0,Speed limit (5km/h),0,Speed limit
3,/Users/thomas/.cache/kagglehub/datasets/ahemat...,0,Speed limit (5km/h),0,Speed limit
4,/Users/thomas/.cache/kagglehub/datasets/ahemat...,0,Speed limit (5km/h),0,Speed limit



- 70% training
- 15% validation
- 15% testing

In [93]:
#double checking that all the unknown classes are gone from our data
print("Unknown rows remaining:", image_df["ClassId"].isin(unknown_ids).sum())
print("Grouped labels present:", sorted(image_df["GroupedClassId"].unique()))

Unknown rows remaining: 0
Grouped labels present: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


In [94]:
train_df, temp_df = train_test_split(
    image_df,
    test_size=0.30,
    random_state=SEED,
    stratify=image_df["GroupedClassId"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["GroupedClassId"],
)

print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

split_counts = pd.DataFrame({
    "train": train_df["GroupedName"].value_counts(),
    "validation": val_df["GroupedName"].value_counts(),
    "test": test_df["GroupedName"].value_counts(),
}).fillna(0).astype(int)

split_counts

Train images: 2709
Validation images: 580
Test images: 581


,train,validation,test
GroupedName,,,
Prohibited / No sign,881,188,189
Warning / Hazard,743,159,160
Speed limit,722,155,155
Mandatory / Direction,363,78,77
